In [1]:
import pandas as pd

df_ground_truth = pd.read_csv('data/ground_truth-new.csv')
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents_llm = []

for i in documents:
    if i["course"]=="llm-zoomcamp":
        documents_llm.append(i)

index=build_index(documents_llm)

In [46]:
def text_search(query):
    boost_dict={"question":3.0, "section":0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [4]:
q=ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [5]:
doc_id=q["document"]
results=text_search(query=q["question"])

In [6]:
for d in results:
    print(f'{d["doc_id"]}=={doc_id}: {d["doc_id"]==doc_id}')

74eb249bbf==74eb249bbf: True
0fab61eca2==74eb249bbf: False
610ccb23c0==74eb249bbf: False
977bf7786c==74eb249bbf: False
04919992b3==74eb249bbf: False


In [7]:
relevance = []

for d in results:
    relevance.append(int(d["doc_id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [8]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [9]:
q2=ground_truth[10]
print(q2["question"])
compute_relevance_text(q2)

How do I join the Office Hours or live workshop if I don’t have the Zoom link?


[1, 0, 0, 0, 0]

In [10]:
q3=ground_truth[80]
print(q3["question"])
compute_relevance_text(q3)

How much do I need to spend to start using the OpenAI API?


[1, 0, 0, 0, 0]

In [11]:
from tqdm.auto import tqdm

def compute_compute_relevance_total_text(ground_truth):
    relevance_total=[]

    for q in tqdm(ground_truth):
        relevance=compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [12]:
gr_truth_sample=ground_truth[:15]
relevance_total_sample=compute_compute_relevance_total_text(gr_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [13]:
relevance_total_sample

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

Rewriting the functions

In [14]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["doc_id"] == doc_id))

    return relevance

In [15]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [16]:
relevance_total=compute_relevance_total(gr_truth_sample,text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [48]:
relevance_total=compute_relevance_total(ground_truth,text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

In [49]:
relevance_total[:15]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

Metrics - Hit Rate

In [50]:
example = relevance_total[:15]

cnt=0
for i in example:
    if 1 in i:
        cnt+=1

cnt

13

In [51]:
hr= (cnt/len(example))*100
print(f'Hit Rate: {hr}%')

Hit Rate: 86.66666666666667%


In [52]:
def hit_rate(relevance):
    cnt=0

    for i in relevance:
        if 1 in i:
            cnt+=1
    
    return cnt/len(relevance)

In [53]:
hit_rate(example)

0.8666666666666667

In [54]:
hit_rate(relevance_total)

0.6506329113924051

Mean Reciprocal Rank (MRR)

In [55]:
example[6]

[0, 1, 0, 0, 0]

In [68]:
total_score=0.0

for line in example:
    for pos in range(len(line)):
        if line[pos]==1:
            score=1/(pos+1)
            total_score = total_score+score
            break

total_score

10.666666666666666

In [57]:
mrr=total_score/len(example)

print(f'The MRR is: {mrr}')

The MRR is: 0.711111111111111


In [69]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for pos in range(len(line)):
            if line[pos] == 1:
                score=1/(pos+1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [70]:
mrr(relevance_total)

0.5734177215189874

Putting it together

In [60]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [61]:
evaluate(ground_truth,text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.6506329113924051, 'mrr': 0.5734177215189874}

Testing with different boost values for question in search_function

In [62]:
def search_boost(query, question_boost):
    boost_dict={"question":question_boost, "section":0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [67]:
for i in [0.5,1.0,1.5,2.0,2.5,3.0,5.0,10.0]:
    result= evaluate(
        ground_truth,
        lambda query, i=i:search_boost(query, i)
    )
    print(f'boost={i}: {result}')

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.6911392405063291, 'mrr': 0.5914345991561182}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.6911392405063291, 'mrr': 0.6011392405063292}


  0%|          | 0/395 [00:00<?, ?it/s]

...

boost=3.0: {'hit_rate': 0.6506329113924051, 'mrr': 0.5734177215189874}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.640506329113924, 'mrr': 0.5578902953586499}


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.6303797468354431, 'mrr': 0.5364556962025318}


In [71]:
def search_boosts(query, ques_boost, answ_boost, sect_boost):
    boost_dict={
        "question": ques_boost,
        "answer": answ_boost,
        "section":sect_boost
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [72]:
results = []

for qboost in [1.0, 2.0, 5.0]:
    for aboost in [1.0, 2.0, 4.0, 10.0]:
        for sboost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={qboost},"
                f" answer_boost={aboost},"
                f" section_boost={sboost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, ques_boost=qboost, answ_boost=aboost, sect_boost=sboost: search_boosts(
                    query,
                    ques_boost,
                    answ_boost,
                    sect_boost
                )
            )

            results.append({
                "question": qboost,
                "answer": aboost,
                "section": sboost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/395 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/395 [00:00<?, ?it/s]

In [75]:
df_results=pd.DataFrame(results)
df_results.sort_values("mrr",ascending=False).head(10)

,question,answer,section,hit_rate,mrr
34,5.0,10.0,0.2,0.754430,0.651055
18,2.0,4.0,0.1,0.754430,0.650633
3,1.0,2.0,0.1,0.751899,0.649705
35,5.0,10.0,0.5,0.751899,0.649705
19,2.0,4.0,0.2,0.751899,0.649705
33,5.0,10.0,0.1,0.749367,0.648354
4,1.0,2.0,0.2,0.744304,0.646751
20,2.0,4.0,0.5,0.746835,0.645021
7,1.0,4.0,0.2,0.754430,0.635654
6,1.0,4.0,0.1,0.754430,0.635021


In [76]:
def text_search_vf(query):
    boost_dict = {
        "question": 5.0,
        "answer": 10.0,
        "section": 0.2,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [77]:
evaluate(ground_truth,text_search_vf)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.7544303797468355, 'mrr': 0.6510548523206751}